## Setup

In [33]:
from huggingface_hub import HfApi
import pandas as pd
import time


api = HfApi()
hf_langs = pd.read_csv('data/hf_langs.csv')

In [40]:
first_10 = ['Hausa', 'Yoruba', 'Chichewa', 'Igbo', 'Oromo',
            'Zulu', 'Xhosa', 'Kinyarwanda', 'Shona', 'Amharic']

second_10 = ['Dari', 'Pashto', 'Ganda', 'Mossi', 'Bemba (Zambia)',
             'Wolof', 'Lingala', 'Bambara', 'Twi', 'Malagasy']

indic_22 = ['Assamese', 'Bengali', 'Bodo (India)', 'Dogri (macrolanguage)', 'Gujarati',
            'Kashmiri', 'Kannada', 'Konkani (macrolanguage)', 'Maithili', 'Malayalam',
            'Manipuri', 'Marathi', 'Nepali (macrolanguage)', 'Odia', 'Panjabi', 'Santali',
            'Sindhi', 'Tamil', 'Telugu', 'Urdu']

## Scrape

*target_langs* need to start with a capital letter

In [35]:
target_langs = indic_22

unwanted_models = ['bloom']

The *mode* argument can either be:

1. 'a' for append mode, appends new models (if any) to the already existing models in the output file
2. 'w' for overwrite mode, rewrites the whole file based on the current *target_langs* list

There is no way to append to an empty file, to use the append mode the output file has to at least have a header

In [36]:
def update_llms_csv(target_langs, unwanted_models, mode='w'):
    hf_langs = pd.read_csv('data/hf_langs.csv')
    target_codes = []
    models_found = []

    for l in target_langs:
        l_codes = hf_langs[hf_langs['language']==l]['code']
        target_codes += list(l_codes)
    print(f'Targeting these language codes: {target_codes}')

    for code in target_codes:
        models = api.list_models(task='text-generation', language=code)
        for i in models:
            supported_langs = [lang for lang in i.tags if lang in list(hf_langs.code)]
            models_found.append([i.id, f'https://huggingface.co/{i.id}', i.downloads, i.likes,
                            len(supported_langs), supported_langs])
            
    models_found = pd.DataFrame(models_found, columns=['llm', 'link', 'n_downloads', 'n_likes', 
                                                       'n_supported_langs', 'supported_langs'])
    models_found.to_csv('output/llms.csv', index=False, header=mode=='w', mode=mode)
    rm_duplicates = pd.read_csv('output/llms.csv').drop_duplicates()
    rm_unwanted = rm_duplicates[~rm_duplicates['llm'].str.contains('|'.join(unwanted_models))]
    rm_unwanted.to_csv('output/llms.csv', index=False, header=True, mode='w')

In [37]:
update_llms_csv(target_langs, unwanted_models, mode='a')

Targeting these language codes: ['as', 'asm', 'bn', 'ben', 'brx', 'doi', 'gu', 'guj', 'hi', 'hin', 'ks', 'kas', 'kn', 'kan', 'kok', 'mai', 'ml', 'mal', 'mni', 'mr', 'mar', 'nep', 'ory', 'pa', 'pan', 'sat', 'sd', 'snd', 'ta', 'tam', 'te', 'tel', 'ur', 'urd']


## Make language first column

In [38]:
def l_first():
    llms = pd.read_csv('output/llms.csv')
    langs = pd.DataFrame(columns=['language', 'hf_codes', 'n_models', 'model-link'])

    langs['language'] = target_langs
    for l in target_langs:
        l_codes = list(hf_langs[hf_langs['language']==l]['code'])
        langs.loc[langs.language == l, 'hf_codes'] = f'{l_codes}'
        models = llms[llms['supported_langs'].str.contains('|'.join(l_codes))].llm
        print(models)
        model_link = [f'{m}: https://huggingface.co/{m}' for m in models]
        langs.loc[langs.language == l, 'n_models'] = len(model_link)
        langs.loc[langs.language == l, 'model-link'] = '\n'.join(model_link)

        langs.to_csv('output/languages.csv', index=False, header=True, mode='w')

In [42]:
l_first()

1                                      LLaMAX/LLaMAX2-7B
2                                      LLaMAX/LLaMAX3-8B
5           tangledgroup/tangled-llama-33m-32k-base-v0.1
6       tangledgroup/tangled-llama-33m-32k-instruct-v0.1
7            tangledgroup/tangled-llama-p-128k-base-v0.1
                              ...                       
210                       goldfish-models/asm_beng_100mb
211                        goldfish-models/asm_beng_10mb
212                         goldfish-models/asm_beng_5mb
1830                       goldfish-models/kas_deva_full
1831                        goldfish-models/kas_deva_5mb
Name: llm, Length: 75, dtype: object
1                                     LLaMAX/LLaMAX2-7B
2                                     LLaMAX/LLaMAX3-8B
4                         lightblue/kurage-multilingual
5          tangledgroup/tangled-llama-33m-32k-base-v0.1
6      tangledgroup/tangled-llama-33m-32k-instruct-v0.1
                             ...                       
